Collections
Unique record IDs
Dense vectors
Sparse vectors
Multiple named vectors
JSON payloads
Metadata indexes
Insert/update/upsert/delete
Filtering
Persistence
Snapshots
Replication and sharding
HTTP/gRPC server

Qdrant Database
│
├── Collection: company-documents
│     │
│     ├── Point 1
│     │     ├── ID
│     │     ├── Dense vector
│     │     └── Payload
│     │
│     ├── Point 2
│     │     ├── ID
│     │     ├── Dense vector
│     │     └── Payload
│     │
│     └── Point 3
│
├── Vector index
│     └── HNSW / related retrieval indexes
│
├── Payload index
│     └── category, source, page, user_id...
│
└── Storage
      ├── Memory
      └── Disk

In [1]:
from __future__ import annotations
from pathlib import Path
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Sunny\AppData\Local\Temp\ipykernel_30528\583762284.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

In [13]:
COLLECTION_NAME = "company-policy-rag"

In [16]:
# 1. Load PDF
loader = PyPDFLoader("D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-34-25-July-2026\\data\\llama2-research-paper.pdf")
pages = loader.load()

print("PDF pages loaded:", len(pages))


PDF pages loaded: 77


In [18]:
# 2. Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=120,
)
chunks = text_splitter.split_documents(pages)
print("Chunks created:", len(chunks))

Chunks created: 174


In [19]:
# 3. Add useful metadata
for chunk_index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_index
    chunk.metadata["filename"] = "llama2-research-paper.pdf"

In [21]:
# ---------------------------------------------------
# 2. Embedding model
# ---------------------------------------------------
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")
dimension = len(
    embeddings.embed_query("dimension check")
)
print("Embedding dimension:", dimension)

dimension = len(
    embeddings.embed_query("dimension check")
)


Embedding dimension: 3072


In [ ]:
# # 5. Local Qdrant
# client = QdrantClient(
#     path=str(Path(__file__).parent / "qdrant_data")
# )


In [22]:
import os
from dotenv import load_dotenv
load_dotenv()
qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_cluster_endpoint = os.getenv("QDRANT_Cluster_Endpoint")


In [24]:
client = QdrantClient(api_key=qdrant_api_key, url=qdrant_cluster_endpoint)

In [25]:
# 6. Create collection
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=dimension,
            distance=models.Distance.COSINE,
        ),
    )

In [26]:
# 7. LangChain Qdrant vector store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)


In [27]:
# 8. Use deterministic or stable IDs in production
chunk_ids = [
    str(uuid4())
    for _ in chunks
]

In [28]:
# 9. Add documents
inserted_ids = vector_store.add_documents(
    documents=chunks,
    ids=chunk_ids,
)

print("Inserted chunks:", len(inserted_ids))

Inserted chunks: 174


In [29]:
query = "What is the annual leave policy?"

results = vector_store.similarity_search(
    query=query,
    k=4,
)

for rank, document in enumerate(results, start=1):
    print(f"\nResult {rank}")
    print("Content:", document.page_content)
    print("Metadata:", document.metadata)


Result 1
Content: approve it. If the answer could not be approved without major changes, the reviewers were asked to reject it
and write the feedback necessary to improve it.
A.5.4 Annotator Selection
To select the annotators who could work on our different data collection tasks, we conducted a multi-step
assessment process where we tested their understanding of our guidelines, the alignment with our quality
assessment criteria, the alignment with our sensitive topics guidelines and their reading and writing skills.
The process included 4 tests:
• Thefirsttestconsistsof3sectionsoftestingtoevaluategrammar,readingcomprehensionandwriting
style. Each section is timed and the test should take a total of 50 minutes to complete. A candidate
must score 90% on part I to continue on to parts II and III, and an average score of 4 on part II and III
to pass the test.
• The second test consisted of 42 questions split into sensitive topics alignment, answer ranking and
two examples of answer writin

In [30]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

context_documents = retriever.invoke(
    "What benefits are available to employees?"
)

context = "\n\n".join(
    document.page_content
    for document in context_documents
)

print(context)

Aroma Mahendru, Joshua Maynez, Vedant Misra, Maysam Moussalem, Zachary Nado, John Nham, Eric
Ni, Andrew Nystrom, Alicia Parrish, Marie Pellat, Martin Polacek, Alex Polozov, Reiner Pope, Siyuan Qiao,
Emily Reif, Bryan Richter, Parker Riley, Alex Castro Ros, Aurko Roy, Brennan Saeta, Rajkumar Samuel,
Renee Shelby, Ambrose Slone, Daniel Smilkov, David R. So, Daniel Sohn, Simon Tokumine, Dasha Valter,
Vijay Vasudevan, Kiran Vodrahalli, Xuezhi Wang, Pidong Wang, Zirui Wang, Tao Wang, John Wieting,
Yuhuai Wu, Kelvin Xu, Yunhan Xu, Linting Xue, Pengcheng Yin, Jiahui Yu, Qiao Zhang, Steven Zheng,
Ce Zheng, Weikang Zhou, Denny Zhou, Slav Petrov, and Yonghui Wu. Palm 2 technical report, 2023.
Amanda Askell, Yuntao Bai, Anna Chen, Dawn Drain, Deep Ganguli, Tom Henighan, Andy Jones, Nicholas
Joseph, Ben Mann, Nova DasSarma, Nelson Elhage, Zac Hatfield-Dodds, Danny Hernandez, Jackson
Kernion, Kamal Ndousse, Catherine Olsson, Dario Amodei, Tom Brown, Jack Clark, Sam McCandlish, and
Chris Olah. A gen